# 04 - Restoration (Part 3: enhance distorted images, re-measure)

Applies each distortion's course-grounded restoration counterpart (median
filter / frequency-domain deconvolution / bilateral filtering + interpolation)
and re-runs all 4 tasks, mirroring 03's sweep structure so distorted vs.
restored performance is directly comparable. The clean baseline (03's
zero-distortion reference point) lives in `02_clean_baseline.csv`.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        # Runtime already had this repo cloned from an earlier cell run in this
        # session - pull so we don't keep running against a stale checkout.
        get_ipython().system(f"git -C {REPO_DIR} pull")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

from ipproj import config
from ipproj.datasets import kitti, kitti_flow
from ipproj.datasets.kitti import read_image
from ipproj.datasets.kitti_flow import read_kitti_flow_png
from ipproj.datasets.materialize import materialize_transformed
from ipproj.tasks import feature_matching, optical_flow, object_detection, semantic_segmentation
from ipproj.distortions import REGISTRY as DISTORTIONS
from ipproj.restoration import REGISTRY as RESTORATIONS
from ipproj.metrics.detection_map import per_class_ap_dict
from ipproj.viz.plotting import (
    plot_before_after, plot_detection_boxes, plot_grouped_bar_per_class, plot_image_grid,
    plot_metric_vs_intensity, plot_optical_flow, plot_orb_keypoints, plot_segmentation_mask, save_figure,
)
from ipproj.reporting import save_results_csv

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()
flow_splits = kitti_flow.load_optical_flow_subset()

checkpoint_path = (config.CHECKPOINT_ROOT / "yolo_clean_baseline_path.txt").read_text().strip()
yolo_model = YOLO(checkpoint_path)
segformer_model, segformer_processor = semantic_segmentation.load_pretrained()

distortion_results = pd.read_csv(config.RESULTS_ROOT / "03_distortions.csv")
clean_baseline = pd.read_csv(config.RESULTS_ROOT / "02_clean_baseline.csv").set_index(["task", "metric"])["value"]

## Before/after: distorted vs. restored (strongest intensity)

In [ ]:
def make_restore_fn(distortion_module, level, restoration_module):
    if restoration_module.NAME == "motion_blur":
        return lambda img: restoration_module.restore(distortion_module.distort(img, level), kernel_size=distortion_module.LEVELS[level])
    return lambda img: restoration_module.restore(distortion_module.distort(img, level))


sample_image = read_image(detection_splits["test"][0].image_path)
for name, module in DISTORTIONS.items():
    level = len(module.LEVELS) - 1
    restore_fn = make_restore_fn(module, level, RESTORATIONS[name])
    distorted = module.distort(sample_image, level)
    restored = restore_fn(sample_image)
    fig = plot_before_after(distorted, restored, title_before=f"{name} distorted", title_after=f"{name} restored")
    save_figure(fig, f"04_restoration/before_after_{name}.png")

## Clean vs. distorted vs. restored (sample image, strongest intensity)

In [ ]:
for name, module in DISTORTIONS.items():
    level = len(module.LEVELS) - 1
    restore_fn = make_restore_fn(module, level, RESTORATIONS[name])
    distorted = module.distort(sample_image, level)
    restored = restore_fn(sample_image)
    fig = plot_image_grid(
        [sample_image, distorted, restored],
        titles=["clean", f"{name} distorted (level {level})", f"{name} restored"],
        ncols=3,
    )
    save_figure(fig, f"04_restoration/clean_distorted_restored_{name}.png")

## Per-task comparison across distortions (clean / distorted / restored, strongest intensity)

For each task, one figure with 9 images: 3 distortions (rows) x clean/distorted/restored (columns), each panel showing that task's own output (boxes, mask, ORB keypoints, or flow field) rather than the plain image.

In [ ]:
clean_boxes, clean_classes, _ = object_detection.predict(yolo_model, sample_image)

fig, axes = plt.subplots(len(DISTORTIONS), 3, figsize=(12, 4 * len(DISTORTIONS)))
for row, (name, module) in enumerate(DISTORTIONS.items()):
    level = len(module.LEVELS) - 1
    restore_fn = make_restore_fn(module, level, RESTORATIONS[name])
    distorted = module.distort(sample_image, level)
    restored = restore_fn(sample_image)

    distorted_boxes, distorted_classes, _ = object_detection.predict(yolo_model, distorted)
    restored_boxes, restored_classes, _ = object_detection.predict(yolo_model, restored)

    plot_detection_boxes(sample_image, clean_boxes, clean_classes, ax=axes[row, 0])
    plot_detection_boxes(distorted, distorted_boxes, distorted_classes, ax=axes[row, 1])
    plot_detection_boxes(restored, restored_boxes, restored_classes, ax=axes[row, 2])
    axes[row, 0].set_title(f"{name}: clean")
    axes[row, 1].set_title(f"{name}: distorted")
    axes[row, 2].set_title(f"{name}: restored")

fig.suptitle("Object detection: clean / distorted / restored per distortion")
fig.tight_layout()
save_figure(fig, "04_restoration/task_comparison_object_detection.png")

In [ ]:
seg_sample_image = read_image(segmentation_splits["test"][0].image_path)
clean_mask = semantic_segmentation.predict(segformer_model, segformer_processor, seg_sample_image)

fig, axes = plt.subplots(len(DISTORTIONS), 3, figsize=(12, 4 * len(DISTORTIONS)))
for row, (name, module) in enumerate(DISTORTIONS.items()):
    level = len(module.LEVELS) - 1
    restore_fn = make_restore_fn(module, level, RESTORATIONS[name])
    distorted = module.distort(seg_sample_image, level)
    restored = restore_fn(seg_sample_image)

    distorted_mask = semantic_segmentation.predict(segformer_model, segformer_processor, distorted)
    restored_mask = semantic_segmentation.predict(segformer_model, segformer_processor, restored)

    plot_segmentation_mask(seg_sample_image, clean_mask, ax=axes[row, 0])
    plot_segmentation_mask(distorted, distorted_mask, ax=axes[row, 1])
    plot_segmentation_mask(restored, restored_mask, ax=axes[row, 2])
    axes[row, 0].set_title(f"{name}: clean")
    axes[row, 1].set_title(f"{name}: distorted")
    axes[row, 2].set_title(f"{name}: restored")

fig.suptitle("Semantic segmentation: clean / distorted / restored per distortion")
fig.tight_layout()
save_figure(fig, "04_restoration/task_comparison_semantic_segmentation.png")

In [ ]:
clean_keypoints, _ = feature_matching.detect_and_describe(sample_image)

fig, axes = plt.subplots(len(DISTORTIONS), 3, figsize=(12, 4 * len(DISTORTIONS)))
for row, (name, module) in enumerate(DISTORTIONS.items()):
    level = len(module.LEVELS) - 1
    restore_fn = make_restore_fn(module, level, RESTORATIONS[name])
    distorted = module.distort(sample_image, level)
    restored = restore_fn(sample_image)

    distorted_keypoints, _ = feature_matching.detect_and_describe(distorted)
    restored_keypoints, _ = feature_matching.detect_and_describe(restored)

    plot_orb_keypoints(sample_image, clean_keypoints, ax=axes[row, 0])
    plot_orb_keypoints(distorted, distorted_keypoints, ax=axes[row, 1])
    plot_orb_keypoints(restored, restored_keypoints, ax=axes[row, 2])
    axes[row, 0].set_title(f"{name}: clean")
    axes[row, 1].set_title(f"{name}: distorted")
    axes[row, 2].set_title(f"{name}: restored")

fig.suptitle("Feature matching (ORB keypoints): clean / distorted / restored per distortion")
fig.tight_layout()
save_figure(fig, "04_restoration/task_comparison_feature_matching.png")

In [ ]:
flow_sample = flow_splits["test"][0]
flow_frame1 = read_image(flow_sample.frame1_path)
flow_frame2_clean = read_image(flow_sample.frame2_path)
clean_flow = optical_flow.compute_flow(flow_frame1, flow_frame2_clean)

fig, axes = plt.subplots(len(DISTORTIONS), 3, figsize=(12, 4 * len(DISTORTIONS)))
for row, (name, module) in enumerate(DISTORTIONS.items()):
    level = len(module.LEVELS) - 1
    restore_fn = make_restore_fn(module, level, RESTORATIONS[name])
    distorted_frame2 = module.distort(flow_frame2_clean, level)
    restored_frame2 = restore_fn(flow_frame2_clean)

    distorted_flow = optical_flow.compute_flow(flow_frame1, distorted_frame2)
    restored_flow = optical_flow.compute_flow(flow_frame1, restored_frame2)

    plot_optical_flow(clean_flow, ax=axes[row, 0])
    plot_optical_flow(distorted_flow, ax=axes[row, 1])
    plot_optical_flow(restored_flow, ax=axes[row, 2])
    axes[row, 0].set_title(f"{name}: clean")
    axes[row, 1].set_title(f"{name}: distorted")
    axes[row, 2].set_title(f"{name}: restored")

fig.suptitle("Optical flow (frame2 distorted/restored): clean / distorted / restored per distortion")
fig.tight_layout()
save_figure(fig, "04_restoration/task_comparison_optical_flow.png")

## Restoration sweep (mirrors 03's structure)

In [ ]:
restoration_results = []
detection_per_class_restored = {}
segmentation_per_class_restored = {}

for name, module in DISTORTIONS.items():
    restoration_module = RESTORATIONS[name]
    for level in range(len(module.LEVELS)):
        restore_fn = make_restore_fn(module, level, restoration_module)
        out_dir = config.RESTORED_ROOT / name / f"level_{level}"

        restored_detection = materialize_transformed(detection_splits["test"], restore_fn, out_dir / "detection")
        detection_metrics = object_detection.evaluate(yolo_model, restored_detection)

        restored_segmentation = materialize_transformed(segmentation_splits["test"], restore_fn, out_dir / "segmentation")
        segmentation_metrics = semantic_segmentation.evaluate(segformer_model, segformer_processor, restored_segmentation)

        if level == len(module.LEVELS) - 1:
            detection_per_class_restored[name] = per_class_ap_dict(detection_metrics)
            segmentation_per_class_restored[name] = segmentation_metrics.tolist()

        match_accuracies = []
        good_match_ratios = []
        for sample in detection_splits["test"][:20]:
            clean = read_image(sample.image_path)
            _, _, _, accuracy, good_match_ratio = feature_matching.match(clean, restore_fn(clean))
            match_accuracies.append(accuracy)
            good_match_ratios.append(good_match_ratio)

        epe_values = []
        fl_error_values = []
        for sample in flow_splits["test"]:
            frame1 = read_image(sample.frame1_path)
            frame2 = read_image(sample.frame2_path)
            gt_flow, valid = read_kitti_flow_png(sample.flow_gt_path)
            flow_metrics = optical_flow.evaluate(frame1, restore_fn(frame2), gt_flow, valid)
            epe_values.append(flow_metrics["epe"])
            fl_error_values.append(flow_metrics["fl_error"])

        restoration_results.append({
            "distortion": name,
            "level": level,
            "match_accuracy": float(np.mean(match_accuracies)),
            "good_match_ratio": float(np.mean(good_match_ratios)),
            "epe": float(np.mean(epe_values)),
            "fl_error": float(np.mean(fl_error_values)),
            "map": float(detection_metrics["map"]),
            "map_50": float(detection_metrics["map_50"]),
            "map_75": float(detection_metrics["map_75"]),
            "mar_100": float(detection_metrics["mar_100"]),
            "mean_iou": float(segmentation_metrics.mean()),
        })

restoration_df = pd.DataFrame(restoration_results)
restoration_df.to_csv(config.RESULTS_ROOT / "04_restoration.csv", index=False)
save_results_csv(restoration_df, "04_restoration/summary.csv")
restoration_df

## Per-class AP/IoU: distorted vs. restored (strongest level)

In [ ]:
detection_per_class_distorted = {}
segmentation_per_class_distorted = {}

for name, module in DISTORTIONS.items():
    level = len(module.LEVELS) - 1
    distort_fn = lambda img, m=module, l=level: m.distort(img, l)
    out_dir = config.DISTORTED_ROOT / name / f"level_{level}"

    distorted_detection = materialize_transformed(detection_splits["test"], distort_fn, out_dir / "detection")
    detection_per_class_distorted[name] = per_class_ap_dict(object_detection.evaluate(yolo_model, distorted_detection))

    distorted_segmentation = materialize_transformed(segmentation_splits["test"], distort_fn, out_dir / "segmentation")
    segmentation_per_class_distorted[name] = semantic_segmentation.evaluate(
        segformer_model, segformer_processor, distorted_segmentation
    ).tolist()

for name, module in DISTORTIONS.items():
    level = len(module.LEVELS) - 1
    detection_series = {
        "distorted": [detection_per_class_distorted[name].get(c, 0.0) for c in config.KITTI_DETECTION_CLASSES],
        "restored": [detection_per_class_restored[name].get(c, 0.0) for c in config.KITTI_DETECTION_CLASSES],
    }
    fig = plot_grouped_bar_per_class(
        config.KITTI_DETECTION_CLASSES, detection_series, ylabel="AP",
        title=f"Detection per-class AP: distorted vs restored ({name}, level {level + 1}/{len(module.LEVELS)})",
    )
    save_figure(fig, f"04_restoration/{name}_detection_per_class_ap.png")

    segmentation_series = {
        "distorted": segmentation_per_class_distorted[name],
        "restored": segmentation_per_class_restored[name],
    }
    fig = plot_grouped_bar_per_class(
        config.CITYSCAPES_TRAINID_LABELS, segmentation_series, ylabel="IoU",
        title=f"Segmentation per-class IoU: distorted vs restored ({name}, level {level + 1}/{len(module.LEVELS)})",
    )
    save_figure(fig, f"04_restoration/{name}_segmentation_per_class_iou.png")

## Distorted vs. restored comparison

In [ ]:
metric_labels = [
    ("map", "mAP"), ("map_50", "mAP@0.5"), ("mean_iou", "mean IoU"),
    ("epe", "EPE"), ("fl_error", "Fl-error"), ("match_accuracy", "match accuracy"),
]

for name in DISTORTIONS:
    distorted_subset = distortion_results[distortion_results["distortion"] == name].sort_values("level")
    restored_subset = restoration_df[restoration_df["distortion"] == name].sort_values("level")
    for metric_col, ylabel in metric_labels:
        fig = plot_metric_vs_intensity(
            distorted_subset["level"].tolist(),
            {"distorted": distorted_subset[metric_col].tolist(), "restored": restored_subset[metric_col].tolist()},
            xlabel="distortion intensity level", ylabel=ylabel, title=f"{name}: {ylabel} - distorted vs restored",
        )
        save_figure(fig, f"04_restoration/{name}_{metric_col}.png")

## Summary: clean vs. distorted vs. restored, per task (strongest severity)

One grouped bar chart per task: a bar per pipeline stage (clean / distorted /
restored), grouped by distortion type, at each distortion's strongest tested
level, with each bar labeled with its numeric value. The clean bar is the
same `02_clean_baseline.csv` value repeated across all three distortion-type
groups. Unlike the line plots above (which show the full severity sweep),
this is a single-glance snapshot at the worst case. Optical flow's EPE is an
error metric (lower is better) - a taller restored bar than distorted means
restoration made it worse, opposite of the other three charts.

In [ ]:
summary_bar_plots = [
    ("match_accuracy", ("feature_matching", "match_accuracy"), "match accuracy", "ORB feature matching"),
    ("epe", ("optical_flow", "epe"), "EPE (lower is better)", "Optical flow (Farneback)"),
    ("map", ("object_detection", "map"), "mAP@0.5:0.95", "YOLOv8 object detection"),
    ("mean_iou", ("semantic_segmentation", "mean_iou"), "mean IoU", "SegFormer semantic segmentation"),
]

labels = list(DISTORTIONS.keys())
for metric_col, baseline_key, ylabel, title in summary_bar_plots:
    clean_value = float(clean_baseline.loc[baseline_key])
    distorted_values, restored_values = [], []
    for name, module in DISTORTIONS.items():
        level = len(module.LEVELS) - 1
        distorted_row = distortion_results[(distortion_results["distortion"] == name) & (distortion_results["level"] == level)]
        restored_row = restoration_df[(restoration_df["distortion"] == name) & (restoration_df["level"] == level)]
        distorted_values.append(float(distorted_row[metric_col].iloc[0]))
        restored_values.append(float(restored_row[metric_col].iloc[0]))

    series = {
        "clean": [clean_value] * len(labels),
        "distorted": distorted_values,
        "restored": restored_values,
    }
    fig = plot_grouped_bar_per_class(
        labels, series, ylabel=ylabel,
        title=f"{title}: clean vs. distorted vs. restored (strongest severity)",
        show_values=True,
    )
    save_figure(fig, f"04_restoration/summary_{metric_col}_bars.png")

## Sync figures/results back to GitHub

`figures/` and `results/` only exist inside this run's throwaway Colab clone
(see the setup cell) - back them up to Drive and push straight to GitHub so
the README's plots/tables actually update. Requires a one-time `GITHUB_TOKEN`
secret in Colab (see this repo's README, "Running on Colab").

In [ ]:
from ipproj.colab_sync import sync_outputs_to_drive, git_commit_and_push

if IN_COLAB:
    sync_outputs_to_drive()
    git_commit_and_push("Sync figures/results from notebook 04 (restoration) run")